# Two Python ML APIs: scikit-learn and PyTorch

Companion to HTB Academy Module 292, Section 4. HTB says its API snippets are *illustrative* and need not be run as one program. This small, self-contained **teaching adaptation** supplies synthetic data so the key calls can be checked safely. It is not the later spam, anomaly, or malware exercise. Run with the **AI** kernel.

## Scikit-learn: split, scale, fit, predict

The six rows below each have two numeric features and a binary class label. This is deliberately tiny and artificial: evaluation is an API check, **not** an estimate of real-world accuracy. We split first. A Pipeline then fits StandardScaler using only the training rows and feeds scaled rows to LogisticRegression. Standardization is z = (x − μ) / σ, read **z equals x minus mu, divided by sigma**. It measures how many training standard deviations x lies from the training mean; e.g. (14 − 10) / 2 = 2.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# X has shape (6 examples, 2 features); y has one label per example.
X = np.array([[0.0, 0.0], [0.0, 0.2], [0.2, 0.0],
              [1.0, 1.0], [1.0, 0.8], [0.8, 1.0]])
y = np.array([0, 0, 0, 1, 1, 1])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=2, stratify=y, random_state=292
)

# fit learns the scaler's training mean/spread, then the classifier's weights.
pipeline = make_pipeline(StandardScaler(), LogisticRegression(C=1.0))
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)  # Apply the trained scaler before prediction.
accuracy = accuracy_score(y_test, predictions)
print('train/test shapes:', X_train.shape, X_test.shape)
print('true/predicted:', y_test.tolist(), predictions.tolist())
print('tiny-demo accuracy:', accuracy)
assert X_train.shape == (4, 2) and X_test.shape == (2, 2)
assert set(predictions).issubset({0, 1})


## PyTorch: one forward/backward/update cycle

The next cell uses a two-input, two-class linear layer rather than HTB's 784→128→10 network, solely to keep shapes and memory easy to inspect. Four examples of two features yield logits of shape (4, 2): one raw score per class per row. CrossEntropyLoss expects these **raw logits** and integer labels, so we do *not* put Softmax inside the training model. The update formula is θ_new = θ_old − η∇_θL, read **theta new equals theta old minus eta times the gradient of L with respect to theta**. In ordinary words, take a small step opposite the loss slope. For θ = 3, eta = 0.1, and gradient = 4, the new value is 2.6.

In [ ]:
import torch
from torch import nn

torch.manual_seed(292)  # Teaching addition: reproducible initial weights.
features = torch.tensor([[0.0, 0.0], [0.0, 1.0],
                         [1.0, 0.0], [1.0, 1.0]])  # Shape (4, 2).
targets = torch.tensor([0, 0, 1, 1])  # Four integer class IDs.
model = nn.Linear(2, 2)  # Two input features → two class logits; CPU only.
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

before = model.weight.detach().clone()  # Snapshot for checking the update.
logits = model(features)  # Forward pass: shape (4, 2).
loss = loss_fn(logits, targets)  # Scalar average classification loss.
optimizer.zero_grad()  # Gradients otherwise accumulate across backward calls.
loss.backward()  # Autograd computes d(loss)/d(parameter) through this graph.
optimizer.step()  # SGD applies parameter = parameter - 0.1 * gradient.

assert tuple(logits.shape) == (4, 2)
assert torch.isfinite(loss).item()
assert not torch.equal(before, model.weight.detach())
print('loss before update:', round(loss.item(), 4))
print('weight matrix changed:', not torch.equal(before, model.weight.detach()))


This demonstrates API mechanics, not a validated detector. The small held-out set cannot support a meaningful security claim. In later lessons we will use the actual course datasets and their specified evaluation procedures. Do not load arbitrary model files into PyTorch: treat downloaded weights and serialized objects as untrusted until verified.